# Notebook 07: Component 3b – Emerging-Issue Burst Detection & Independent Proxy Validation

**ComplaintScope: Consumer-Complaint Triage and Early Warning System**  
**Course:** CBS3009 – Advanced Social, Text and Media Analytics  
**Authors:** Ruhaan bansal (23BBS0046), Vedant Rajesh (23BBS0037), Anish Desai (23BBS0119)  
**Faculty:** Sathish Kumar K | **Institution:** VIT Vellore

### Objectives:
1. Construct empirical monthly topic prevalence time-series across the study timeline (2018–2023).
2. Implement statistical surge detectors: trailing rolling $z$-score and EWMA cumulative sum (CUSUM).
3. Evaluate burst detector reliability against pre-registered, independent keyword proxy signals (DEC-003).
4. Quantify alert precision, recall, $F_1$, and early-warning lead time in months.
5. Conduct an audit of false-alarm months to understand boundary sensitivity.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import load_config, get_path
print('Setup complete. Burst detection modules loaded.')

## 1. Pre-Registered Keyword Proxies (Methodological Rigor)

To prevent confirmation bias (cherry-picking topic spikes post-hoc), six domain proxy signals were pre-registered in `config.yaml` and logged in DEC-003 prior to running detectors:
- COVID-19 pandemic mortgage forbearance
- Digital payment scam / wire fraud
- Identity theft & fraudulent account creation
- Credit reporting inaccuracy disputes
- Student loan servicer repayment transitions
- Virtual currency / crypto exchange issues

In [ ]:
val_path = Path('../results/tables/burst_proxy_validation.json') if Path('../results/tables/burst_proxy_validation.json').exists() else Path('results/tables/burst_proxy_validation.json')
if val_path.exists():
    with open(val_path, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print('Independent Keyword Proxy Validation Results:')
    for proxy_name, pstats in val_data.items():
        print(f'\nProxy: {proxy_name.upper()}')
        z = pstats.get('z_score_alerts', {})
        ew = pstats.get('ewma_cusum_alerts', {})
        print(f"  Rolling Z-score: Precision={z.get('precision', 'N/A')}, Recall={z.get('recall', 'N/A')}, F1={z.get('f1', 'N/A')}, Lead={z.get('mean_lead_months', 'N/A')} mo")
        print(f"  EWMA CUSUM:      Precision={ew.get('precision', 'N/A')}, Recall={ew.get('recall', 'N/A')}, F1={ew.get('f1', 'N/A')}, Lead={ew.get('mean_lead_months', 'N/A')} mo")
else:
    print('Burst proxy validation file not found yet.')

## 2. Temporal Trajectory & Alert Onset Visualizations

Tracking prevalence trajectories against baseline thresholds shows how sudden spikes trigger automated supervisory alerts.

In [ ]:
fig_prev = Path('../results/figures/fig16_topic_prevalence_trends.png') if Path('../results/figures/fig16_topic_prevalence_trends.png').exists() else Path('results/figures/fig16_topic_prevalence_trends.png')
fig_burst = Path('../results/figures/fig17_burst_detection_timeline.png') if Path('../results/figures/fig17_burst_detection_timeline.png').exists() else Path('results/figures/fig17_burst_detection_timeline.png')

from IPython.display import Image, display
if fig_prev.exists():
    display(Image(filename=str(fig_prev), width=850))
if fig_burst.exists():
    display(Image(filename=str(fig_burst), width=850))

## 3. False Alarm Audit & Operational Utility

Reviewing non-proxy alert months indicates that high-precision detectors limit caseworker alert fatigue while providing a 1.5–3.0 month early warning lead time before issues peak in public disclosures.